<a href="https://colab.research.google.com/github/MariamHazem226/AI-in-Software-Debugging-Research/blob/main/mbpp_experiments/MBPP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install groq datasets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 9.1 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
"""
COMPLETE MBPP EVALUATION (n=30)
Methods 1-12 using shared agents
"""

import re
import time
import os
import json
from groq import Groq
from datasets import load_dataset

MODEL = "llama-3.3-70b-versatile"
# Replace with your Groq API key
API_KEY = "your_actual_key_here"
client = Groq(api_key=API_KEY)

TOTAL = 30
RESULTS_PATH = "/content/drive/MyDrive/results_mbpp.json"

dataset = load_dataset("mbpp", split="test")

print(f"MBPP loaded: {len(dataset)} samples, evaluating first {TOTAL}")
print(f"Model: {MODEL}")
print("=" * 60)


def extract_code(text):
    patterns = [
        r"```python\s*(.*?)```",
        r"```\s*(.*?)```",
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.DOTALL)
        if match:
            return match.group(1).strip()
    return text.strip()


def extract_func_name(test_list):
    match = re.search(r'assert (\w+)\(', test_list[0])
    return match.group(1) if match else "solution"


def safe_api_call(messages, temperature=0.2, max_tokens=400):
    wait = 60
    while True:
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens
            )
            return response.choices[0].message.content
        except Exception as e:
            if "429" in str(e):
                match = re.search(r'try again in (\d+)m([\d.]+)s', str(e))
                if match:
                    wait = int(match.group(1)) * 60 + int(float(match.group(2))) + 5
                print(f"Rate limit hit, waiting {wait} seconds...")
                time.sleep(wait)
                wait = min(wait * 2, 300)
            else:
                raise


def run_tests(code, test_list):
    local_env = {}
    exec(code, local_env)
    for test in test_list:
        exec(test, local_env)
    return True


def save_result(key, value):
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH, "r") as f:
            all_results = json.load(f)
    else:
        all_results = {}
    all_results[key] = value
    with open(RESULTS_PATH, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"Saved: {key} = {value:.1f}%")
    print(f"All results: {all_results}")


def evaluate_method(solve_func, method_name, delay=20):
    print(f"\n{'='*60}")
    print(f"EVALUATING: {method_name}")
    print(f"{'='*60}")
    passed = 0
    for idx, sample in enumerate(dataset.select(range(TOTAL))):
        func_name = extract_func_name(sample["test_list"])
        print(f"\n[{idx+1}/{TOTAL}] {func_name}")
        try:
            result = solve_func(sample["text"], sample["test_list"], func_name)
            if result:
                passed += 1
                print(f"PASS ({passed}/{idx+1})")
            else:
                print(f"FAIL ({passed}/{idx+1})")
        except Exception as e:
            print(f"ERROR: {e}")
        time.sleep(delay)
    accuracy = (passed / TOTAL) * 100
    print(f"\n{method_name} Accuracy: {passed}/{TOTAL} = {accuracy:.1f}%")
    return accuracy

print("Setup complete.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

full/train-00000-of-00001.parquet:   0%|          | 0.00/87.2k [00:00<?, ?B/s]

full/test-00000-of-00001.parquet:   0%|          | 0.00/116k [00:00<?, ?B/s]

full/validation-00000-of-00001.parquet:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/374 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/90 [00:00<?, ? examples/s]

Generating prompt split:   0%|          | 0/10 [00:00<?, ? examples/s]

MBPP loaded: 500 samples, evaluating first 30
Model: llama-3.3-70b-versatile
Setup complete.


In [ ]:
# SHARED AGENTS - used across all methods
# Standard Agents
def planning_agent(problem):
    return safe_api_call(
        messages=[{
            "role": "user",
            "content": f"""
You are a planning agent.

Problem:
{problem}

Return ONLY a numbered step-by-step plan.
No code. No explanation.
"""
        }],
        temperature=0.3,
        max_tokens=300
    )


def coding_agent(problem, plan, func_name):
    return safe_api_call(
        messages=[{
            "role": "user",
            "content": f"""
You are a coding agent.

Problem:
{problem}

Plan:
{plan}

IMPORTANT: The function MUST be named exactly: {func_name}

Return ONLY Python function implementation.
No explanation. No markdown.
"""
        }],
        temperature=0.2,
        max_tokens=400
    )


def debugging_agent(problem, code, error, func_name):
    return safe_api_call(
        messages=[{
            "role": "user",
            "content": f"""
You are a debugging agent.

Fix the code.

Problem:
{problem}

IMPORTANT: The function MUST be named exactly: {func_name}

Code:
{code}

Error:
{error}

Return ONLY corrected Python function.
No explanation.
"""
        }],
        temperature=0.2,
        max_tokens=400
    )


# CODESIM Agents
def codesim_planning_agent(problem, func_name, test_list):
    return safe_api_call(
        messages=[{
            "role": "user",
            "content": f"""
You are a simulation planning agent.

Problem:
{problem}

Function name: {func_name}
Test cases: {test_list}

First, simulate step-by-step what the function should do for each test case.
Then write a numbered plan.

Return ONLY the simulation trace and plan.
No code.
"""
        }],
        temperature=0.3,
        max_tokens=600
    )


def codesim_coding_agent(problem, plan, func_name):
    return safe_api_call(
        messages=[{
            "role": "user",
            "content": f"""
You are a coding agent.

Problem:
{problem}

Simulation and Plan:
{plan}

IMPORTANT: The function MUST be named exactly: {func_name}

Return ONLY the Python function implementation.
No explanation. No markdown.
"""
        }],
        temperature=0.2,
        max_tokens=400
    )


def codesim_debugging_agent(problem, code, error, func_name):
    return safe_api_call(
        messages=[{
            "role": "user",
            "content": f"""
You are a debugging agent.

Problem:
{problem}

IMPORTANT: The function MUST be named exactly: {func_name}

Code:
{code}

Error:
{error}

Simulate the correct behavior first, then return ONLY the corrected Python function.
"""
        }],
        temperature=0.2,
        max_tokens=500
    )

print("Shared agents ready.")

Shared agents ready.


In [ ]:
# Standard Multi-Agent
def standard_ma_solve(problem, test_list, func_name, retries=2):
    try:
        plan = planning_agent(problem)
        code = extract_code(coding_agent(problem, plan, func_name))
        if not code:
            return False
        for attempt in range(retries + 1):
            try:
                run_tests(code, test_list)
                print(f"PASSED (attempt {attempt + 1})")
                return True
            except Exception as e:
                print(f"FAIL (attempt {attempt + 1}): {str(e)[:60]}")
                if attempt < retries:
                    code = extract_code(debugging_agent(problem, code, str(e), func_name))
                    if not code:
                        return False
        return False
    except Exception as e:
        print(f"CRITICAL ERROR: {e}")
        return False


acc_standard_ma = evaluate_method(standard_ma_solve, "Standard Multi-Agent", delay=20)
save_result("standard_ma", acc_standard_ma)


EVALUATING: Standard Multi-Agent

[1/30] remove_Occ
PASSED (attempt 1)
PASS (1/1)

[2/30] sort_matrix
PASSED (attempt 1)
PASS (2/2)

[3/30] count_common
FAIL (attempt 1): 'list' object has no attribute 'values'
FAIL (attempt 2): 'list' object has no attribute 'values'
FAIL (attempt 3): 
FAIL (2/3)

[4/30] find_Volume
PASSED (attempt 1)
PASS (3/4)

[5/30] split_lowerstring
FAIL (attempt 1): 
FAIL (attempt 2): 
FAIL (attempt 3): 
FAIL (3/5)

[6/30] text_lowercase_underscore
FAIL (attempt 1): 
FAIL (attempt 2): 
FAIL (attempt 3): 
FAIL (3/6)

[7/30] square_perimeter
PASSED (attempt 1)
PASS (4/7)

[8/30] remove_dirty_chars
PASSED (attempt 1)
PASS (5/8)

[9/30] test_duplicate
PASSED (attempt 1)
PASS (6/9)

[10/30] is_woodall
FAIL (attempt 1): 
FAIL (attempt 2): 
FAIL (attempt 3): 
FAIL (6/10)

[11/30] multiples_of_num
PASSED (attempt 1)
PASS (7/11)

[12/30] find_first_duplicate
FAIL (attempt 1): 
FAIL (attempt 2): 
FAIL (attempt 3): 
FAIL (7/12)

[13/30] maximum_Sum
PASSED (attempt 1)
PASS

In [ ]:
# Standard Self-Debug
def standard_sd_solve(problem, test_list, func_name, iterations=3):
    try:
        code = extract_code(safe_api_call(
            messages=[{
                "role": "user",
                "content": f"""
Solve this problem.

Problem:
{problem}

IMPORTANT: The function MUST be named exactly: {func_name}

Return ONLY the Python function.
No explanation. No markdown.
"""
            }],
            temperature=0.2,
            max_tokens=400
        ))
        for i in range(iterations):
            try:
                run_tests(code, test_list)
                print(f"PASSED (iteration {i + 1})")
                return True
            except Exception as e:
                print(f"FAIL (iteration {i + 1}): {str(e)[:60]}")
                code = extract_code(safe_api_call(
                    messages=[{
                        "role": "user",
                        "content": f"""
The following code failed.

Problem:
{problem}

IMPORTANT: The function MUST be named exactly: {func_name}

Code:
{code}

Error:
{str(e)}

Fix the code. Return ONLY the corrected Python function.
"""
                    }],
                    temperature=0.2,
                    max_tokens=400
                ))
        return False
    except Exception as e:
        print(f"CRITICAL ERROR: {e}")
        return False


acc_standard_sd = evaluate_method(standard_sd_solve, "Standard Self-Debug", delay=20)
save_result("standard_sd", acc_standard_sd)


EVALUATING: Standard Self-Debug

[1/30] remove_Occ
PASSED (iteration 1)
PASS (1/1)

[2/30] sort_matrix
PASSED (iteration 1)
PASS (2/2)

[3/30] count_common
FAIL (iteration 1): 'list' object has no attribute 'values'
FAIL (iteration 2): 
FAIL (iteration 3): 'list' object has no attribute 'values'
FAIL (2/3)

[4/30] find_Volume
PASSED (iteration 1)
PASS (3/4)

[5/30] split_lowerstring
FAIL (iteration 1): 
FAIL (iteration 2): 
FAIL (iteration 3): 
FAIL (3/5)

[6/30] text_lowercase_underscore
FAIL (iteration 1): 
FAIL (iteration 2): 
FAIL (iteration 3): 
FAIL (3/6)

[7/30] square_perimeter
PASSED (iteration 1)
PASS (4/7)

[8/30] remove_dirty_chars
PASSED (iteration 1)
PASS (5/8)

[9/30] test_duplicate
PASSED (iteration 1)
PASS (6/9)

[10/30] is_woodall
PASSED (iteration 1)
PASS (7/10)

[11/30] multiples_of_num
PASSED (iteration 1)
PASS (8/11)

[12/30] find_first_duplicate
FAIL (iteration 1): 
FAIL (iteration 2): 
FAIL (iteration 3): 
FAIL (8/12)

[13/30] maximum_Sum
PASSED (iteration 1)
P

In [ ]:
# Standard Hybrid
def standard_hybrid_solve(problem, test_list, func_name, retries=3):
    try:
        plan = planning_agent(problem)
        code = extract_code(coding_agent(problem, plan, func_name))
        if not code:
            return False
        for attempt in range(retries):
            try:
                run_tests(code, test_list)
                print(f"PASSED (attempt {attempt + 1})")
                return True
            except Exception as e:
                print(f"FAIL (attempt {attempt + 1}): {str(e)[:60]}")
                if attempt == 0:
                    code = extract_code(debugging_agent(problem, code, str(e), func_name))
                else:
                    code = extract_code(safe_api_call(
                        messages=[{
                            "role": "user",
                            "content": f"""
Fix the following code.

IMPORTANT: The function MUST be named exactly: {func_name}

Error:
{str(e)}

Code:
{code}

Return ONLY the corrected Python function.
"""
                        }],
                        temperature=0.1,
                        max_tokens=400
                    ))
        return False
    except Exception as e:
        print(f"CRITICAL ERROR: {e}")
        return False


acc_standard_hybrid = evaluate_method(standard_hybrid_solve, "Standard Hybrid", delay=20)
save_result("standard_hybrid", acc_standard_hybrid)


EVALUATING: Standard Hybrid

[1/30] remove_Occ
PASSED (attempt 1)
PASS (1/1)

[2/30] sort_matrix
PASSED (attempt 1)
PASS (2/2)

[3/30] count_common
FAIL (attempt 1): 'list' object has no attribute 'values'
FAIL (attempt 2): 
FAIL (attempt 3): 
FAIL (2/3)

[4/30] find_Volume
PASSED (attempt 1)
PASS (3/4)

[5/30] split_lowerstring
FAIL (attempt 1): 
FAIL (attempt 2): 
FAIL (attempt 3): 
FAIL (3/5)

[6/30] text_lowercase_underscore
FAIL (attempt 1): 
FAIL (attempt 2): 
FAIL (attempt 3): 
FAIL (3/6)

[7/30] square_perimeter
PASSED (attempt 1)
PASS (4/7)

[8/30] remove_dirty_chars
PASSED (attempt 1)
PASS (5/8)

[9/30] test_duplicate
PASSED (attempt 1)
PASS (6/9)

[10/30] is_woodall
FAIL (attempt 1): 
FAIL (attempt 2): 
FAIL (attempt 3): 
FAIL (6/10)

[11/30] multiples_of_num
PASSED (attempt 1)
PASS (7/11)

[12/30] find_first_duplicate
FAIL (attempt 1): 
FAIL (attempt 2): 
PASSED (attempt 3)
PASS (8/12)

[13/30] maximum_Sum
PASSED (attempt 1)
PASS (9/13)

[14/30] binary_to_decimal
FAIL (att

In [ ]:
# CODESIM Multi-Agent
def codesim_ma_solve(problem, test_list, func_name, retries=2):
    try:
        plan = codesim_planning_agent(problem, func_name, test_list)
        code = extract_code(codesim_coding_agent(problem, plan, func_name))
        if not code:
            return False
        for attempt in range(retries + 1):
            try:
                run_tests(code, test_list)
                print(f"PASSED (attempt {attempt + 1})")
                return True
            except Exception as e:
                print(f"FAIL (attempt {attempt + 1}): {str(e)[:60]}")
                if attempt < retries:
                    code = extract_code(codesim_debugging_agent(problem, code, str(e), func_name))
                    if not code:
                        return False
        return False
    except Exception as e:
        print(f"CRITICAL ERROR: {e}")
        return False


acc_codesim_ma = evaluate_method(codesim_ma_solve, "CODESIM Multi-Agent", delay=20)
save_result("codesim_ma", acc_codesim_ma)


EVALUATING: CODESIM Multi-Agent

[1/30] remove_Occ
PASSED (attempt 1)
PASS (1/1)

[2/30] sort_matrix
PASSED (attempt 1)
PASS (2/2)

[3/30] count_common
FAIL (attempt 1): 
FAIL (attempt 2): name 'count_common' is not defined
The most common word(s) is/are ['banana'] with 3 occurrences.
FAIL (attempt 3): dictionary update sequence element #0 has length 3; 2 is req
FAIL (2/3)

[4/30] find_Volume
PASSED (attempt 1)
PASS (3/4)

[5/30] split_lowerstring
FAIL (attempt 1): 
FAIL (attempt 2): 
FAIL (attempt 3): 
FAIL (3/5)

[6/30] text_lowercase_underscore
PASSED (attempt 1)
PASS (4/6)

[7/30] square_perimeter
PASSED (attempt 1)
PASS (5/7)

[8/30] remove_dirty_chars
PASSED (attempt 1)
PASS (6/8)

[9/30] test_duplicate
PASSED (attempt 1)
PASS (7/9)

[10/30] is_woodall
PASSED (attempt 1)
PASS (8/10)

[11/30] multiples_of_num
PASSED (attempt 1)
PASS (9/11)

[12/30] find_first_duplicate
PASSED (attempt 1)
PASS (10/12)

[13/30] maximum_Sum
PASSED (attempt 1)
PASS (11/13)

[14/30] binary_to_decimal


In [ ]:
# CODESIM Self-Debug
def codesim_sd_solve(problem, test_list, func_name, iterations=3):
    try:
        response = safe_api_call(
            messages=[{
                "role": "user",
                "content": f"""
Simulate the correct behavior step-by-step for the following problem.
Show the simulation trace, then write the Python function.

Problem:
{problem}

Function name: {func_name}
Test cases: {test_list}

Return the simulation trace AND the Python function.
"""
            }],
            temperature=0.2,
            max_tokens=600
        )
        code = extract_code(response)
        for i in range(iterations):
            try:
                run_tests(code, test_list)
                print(f"PASSED (iteration {i + 1})")
                return True
            except Exception as e:
                print(f"FAIL (iteration {i + 1}): {str(e)[:60]}")
                code = extract_code(safe_api_call(
                    messages=[{
                        "role": "user",
                        "content": f"""
Problem:
{problem}

IMPORTANT: The function MUST be named exactly: {func_name}

Code:
{code}

Error:
{str(e)}

Simulate what went wrong, then return ONLY the corrected Python function.
"""
                    }],
                    temperature=0.2,
                    max_tokens=500
                ))
        return False
    except Exception as e:
        print(f"CRITICAL ERROR: {e}")
        return False


acc_codesim_sd = evaluate_method(codesim_sd_solve, "CODESIM Self-Debug", delay=20)
save_result("codesim_sd", acc_codesim_sd)


EVALUATING: CODESIM Self-Debug

[1/30] remove_Occ
PASSED (iteration 1)
PASS (1/1)

[2/30] sort_matrix
FAIL (iteration 1): unterminated string literal (detected at line 10) (<string>,
PASSED (iteration 2)
PASS (2/2)

[3/30] count_common
FAIL (iteration 1): 
FAIL (iteration 2): 
FAIL (iteration 3): 
FAIL (2/3)

[4/30] find_Volume
PASSED (iteration 1)
PASS (3/4)

[5/30] split_lowerstring
FAIL (iteration 1): 
FAIL (iteration 2): 
FAIL (iteration 3): 
FAIL (3/5)

[6/30] text_lowercase_underscore
PASSED (iteration 1)
PASS (4/6)

[7/30] square_perimeter
PASSED (iteration 1)
PASS (5/7)

[8/30] remove_dirty_chars
PASSED (iteration 1)
PASS (6/8)

[9/30] test_duplicate
FAIL (iteration 1): invalid syntax (<string>, line 1)
PASSED (iteration 2)
PASS (7/9)

[10/30] is_woodall
FAIL (iteration 1): unterminated string literal (detected at line 3) (<string>, 
PASSED (iteration 2)
PASS (8/10)

[11/30] multiples_of_num
FAIL (iteration 1): unterminated string literal (detected at line 4) (<string>, 
PASSE

In [ ]:
# CODESIM Hybrid
def codesim_hybrid_solve(problem, test_list, func_name, retries=3):
    try:
        plan = codesim_planning_agent(problem, func_name, test_list)
        code = extract_code(codesim_coding_agent(problem, plan, func_name))
        if not code:
            return False
        for attempt in range(retries):
            try:
                run_tests(code, test_list)
                print(f"PASSED (attempt {attempt + 1})")
                return True
            except Exception as e:
                print(f"FAIL (attempt {attempt + 1}): {str(e)[:60]}")
                if attempt == 0:
                    code = extract_code(codesim_debugging_agent(problem, code, str(e), func_name))
                else:
                    code = extract_code(safe_api_call(
                        messages=[{
                            "role": "user",
                            "content": f"""
Fix the following code.

IMPORTANT: The function MUST be named exactly: {func_name}

Error:
{str(e)}

Code:
{code}

Return ONLY the corrected Python function.
"""
                        }],
                        temperature=0.1,
                        max_tokens=400
                    ))
        return False
    except Exception as e:
        print(f"CRITICAL ERROR: {e}")
        return False


acc_codesim_hybrid = evaluate_method(codesim_hybrid_solve, "CODESIM Hybrid", delay=20)
save_result("codesim_hybrid", acc_codesim_hybrid)


EVALUATING: CODESIM Hybrid

[1/30] remove_Occ
PASSED (attempt 1)
PASS (1/1)

[2/30] sort_matrix
PASSED (attempt 1)
PASS (2/2)

[3/30] count_common
FAIL (attempt 1): 
FAIL (attempt 2): name 'count_common' is not defined
6
FAIL (attempt 3): too many values to unpack (expected 2)
FAIL (2/3)

[4/30] find_Volume
PASSED (attempt 1)
PASS (3/4)

[5/30] split_lowerstring
FAIL (attempt 1): 
FAIL (attempt 2): 
FAIL (attempt 3): 
FAIL (3/5)

[6/30] text_lowercase_underscore
PASSED (attempt 1)
PASS (4/6)

[7/30] square_perimeter
PASSED (attempt 1)
PASS (5/7)

[8/30] remove_dirty_chars
PASSED (attempt 1)
PASS (6/8)

[9/30] test_duplicate
PASSED (attempt 1)
PASS (7/9)

[10/30] is_woodall
PASSED (attempt 1)
PASS (8/10)

[11/30] multiples_of_num
PASSED (attempt 1)
PASS (9/11)

[12/30] find_first_duplicate
PASSED (attempt 1)
PASS (10/12)

[13/30] maximum_Sum
PASSED (attempt 1)
PASS (11/13)

[14/30] binary_to_decimal
PASSED (attempt 1)
PASS (12/14)

[15/30] find_Product
PASSED (attempt 1)
PASS (13/15)



In [ ]:
# Normal AgentCoder
def normal_agentcoder_solve(problem, test_list, func_name, retries=2):
    try:
        plan = planning_agent(problem)
        code = extract_code(coding_agent(problem, plan, func_name))
        if not code:
            return False
        safe_api_call(
            messages=[{
                "role": "user",
                "content": f"""
Generate 2 edge-case assert statements for the following problem.

Problem:
{problem}

Function name: {func_name}

Return ONLY the assert statements, one per line.
"""
            }],
            temperature=0.3,
            max_tokens=150
        )
        for attempt in range(retries + 1):
            try:
                run_tests(code, test_list)
                print(f"PASSED (attempt {attempt + 1})")
                return True
            except Exception as e:
                print(f"FAIL (attempt {attempt + 1}): {str(e)[:60]}")
                if attempt < retries:
                    code = extract_code(debugging_agent(problem, code, str(e), func_name))
                    if not code:
                        return False
        return False
    except Exception as e:
        print(f"CRITICAL ERROR: {e}")
        return False


acc_normal_agentcoder = evaluate_method(normal_agentcoder_solve, "Normal AgentCoder", delay=20)
save_result("normal_agentcoder", acc_normal_agentcoder)


EVALUATING: Normal AgentCoder

[1/30] remove_Occ
PASSED (attempt 1)
PASS (1/1)

[2/30] sort_matrix
PASSED (attempt 1)
PASS (2/2)

[3/30] count_common
FAIL (attempt 1): 'list' object has no attribute 'items'
FAIL (attempt 2): 'list' object has no attribute 'items'
FAIL (attempt 3): 'list' object has no attribute 'items'
FAIL (2/3)

[4/30] find_Volume
PASSED (attempt 1)
PASS (3/4)

[5/30] split_lowerstring
FAIL (attempt 1): 
FAIL (attempt 2): 
FAIL (attempt 3): 
FAIL (3/5)

[6/30] text_lowercase_underscore
FAIL (attempt 1): 
FAIL (attempt 2): 
FAIL (attempt 3): 
FAIL (3/6)

[7/30] square_perimeter
PASSED (attempt 1)
PASS (4/7)

[8/30] remove_dirty_chars
PASSED (attempt 1)
PASS (5/8)

[9/30] test_duplicate
PASSED (attempt 1)
PASS (6/9)

[10/30] is_woodall
FAIL (attempt 1): 
FAIL (attempt 2): 
FAIL (attempt 3): 
FAIL (6/10)

[11/30] multiples_of_num
PASSED (attempt 1)
PASS (7/11)

[12/30] find_first_duplicate
FAIL (attempt 1): 
FAIL (attempt 2): 
FAIL (attempt 3): 
FAIL (7/12)

[13/30] ma

In [ ]:
# Normal Codex Baseline
def normal_codex_solve(problem, test_list, func_name):
    try:
        code = extract_code(safe_api_call(
            messages=[{
                "role": "user",
                "content": f"""
Write a Python function to solve the following problem.

Problem:
{problem}

IMPORTANT: The function MUST be named exactly: {func_name}

Return ONLY the Python function.
No explanation. No markdown.
"""
            }],
            temperature=0.0,
            max_tokens=400
        ))
        run_tests(code, test_list)
        print("PASSED")
        return True
    except Exception as e:
        print(f"FAIL: {str(e)[:60]}")
        return False


acc_normal_codex = evaluate_method(normal_codex_solve, "Normal Codex Baseline", delay=10)
save_result("normal_codex", acc_normal_codex)


EVALUATING: Normal Codex Baseline

[1/30] remove_Occ
PASSED
PASS (1/1)

[2/30] sort_matrix
PASSED
PASS (2/2)

[3/30] count_common
FAIL: 'list' object has no attribute 'values'
FAIL (2/3)

[4/30] find_Volume
PASSED
PASS (3/4)

[5/30] split_lowerstring
FAIL: 
FAIL (3/5)

[6/30] text_lowercase_underscore
FAIL: 
FAIL (3/6)

[7/30] square_perimeter
PASSED
PASS (4/7)

[8/30] remove_dirty_chars
PASSED
PASS (5/8)

[9/30] test_duplicate
PASSED
PASS (6/9)

[10/30] is_woodall
PASSED
PASS (7/10)

[11/30] multiples_of_num
PASSED
PASS (8/11)

[12/30] find_first_duplicate
FAIL: 
FAIL (8/12)

[13/30] maximum_Sum
PASSED
PASS (9/13)

[14/30] binary_to_decimal
FAIL: int() can't convert non-string with explicit base
FAIL (9/14)

[15/30] find_Product
FAIL: find_Product() takes 1 positional argument but 2 were given
FAIL (9/15)

[16/30] check_k_elements
FAIL: 
FAIL (9/16)

[17/30] remove
PASSED
PASS (10/17)

[18/30] binomial_Coeff
PASSED
PASS (11/18)

[19/30] get_Odd_Occurrence
FAIL: get_Odd_Occurrence() t

In [ ]:
# Normal MGDebugger
def normal_mgdebugger_solve(problem, test_list, func_name, retries=2):
    try:
        code = extract_code(safe_api_call(
            messages=[{
                "role": "user",
                "content": f"""
Write a correct Python function for the following problem.

Problem:
{problem}

IMPORTANT: The function MUST be named exactly: {func_name}

Return ONLY the Python function.
"""
            }],
            temperature=0.2,
            max_tokens=400
        ))
        decomp = safe_api_call(
            messages=[{
                "role": "user",
                "content": f"""
Break the following problem into small logical steps.

Problem:
{problem}

Return ONLY the steps. No code.
"""
            }],
            temperature=0.3,
            max_tokens=200
        )
        for attempt in range(retries + 1):
            try:
                run_tests(code, test_list)
                print(f"PASSED (attempt {attempt + 1})")
                return True
            except Exception as e:
                print(f"FAIL (attempt {attempt + 1}): {str(e)[:60]}")
                if attempt < retries:
                    code = extract_code(safe_api_call(
                        messages=[{
                            "role": "user",
                            "content": f"""
You are a hierarchical debugger.

Problem steps:
{decomp}

IMPORTANT: The function MUST be named exactly: {func_name}

Code:
{code}

Error:
{str(e)}

Fix the minimal part that caused the error.
Return ONLY the corrected Python function.
"""
                        }],
                        temperature=0.2,
                        max_tokens=500
                    ))
        return False
    except Exception as e:
        print(f"CRITICAL ERROR: {e}")
        return False


acc_normal_mgdebugger = evaluate_method(normal_mgdebugger_solve, "Normal MGDebugger", delay=20)
save_result("normal_mgdebugger", acc_normal_mgdebugger)


EVALUATING: Normal MGDebugger

[1/30] remove_Occ
PASSED (attempt 1)
PASS (1/1)

[2/30] sort_matrix
PASSED (attempt 1)
PASS (2/2)

[3/30] count_common
FAIL (attempt 1): 'list' object has no attribute 'keys'
FAIL (attempt 2): 'list' object has no attribute 'items'
FAIL (attempt 3): 'list' object has no attribute 'items'
FAIL (2/3)

[4/30] find_Volume
PASSED (attempt 1)
PASS (3/4)

[5/30] split_lowerstring
FAIL (attempt 1): 
FAIL (attempt 2): 
FAIL (attempt 3): 
FAIL (3/5)

[6/30] text_lowercase_underscore
FAIL (attempt 1): 
FAIL (attempt 2): 
FAIL (attempt 3): 
FAIL (3/6)

[7/30] square_perimeter
PASSED (attempt 1)
PASS (4/7)

[8/30] remove_dirty_chars
PASSED (attempt 1)
PASS (5/8)

[9/30] test_duplicate
PASSED (attempt 1)
PASS (6/9)

[10/30] is_woodall
PASSED (attempt 1)
PASS (7/10)

[11/30] multiples_of_num
PASSED (attempt 1)
PASS (8/11)

[12/30] find_first_duplicate
FAIL (attempt 1): 
FAIL (attempt 2): 
FAIL (attempt 3): 
FAIL (8/12)

[13/30] maximum_Sum
PASSED (attempt 1)
PASS (9/13

In [ ]:
# AgentCoder Reference [19]
def ref_agentcoder_solve(problem, test_list, func_name, retries=2):
    try:
        code = extract_code(safe_api_call(
            messages=[{
                "role": "user",
                "content": f"""
You are a programmer agent.
Write a complete and correct Python function.

Problem:
{problem}

IMPORTANT: The function MUST be named exactly: {func_name}

Return ONLY the Python function.
"""
            }],
            temperature=0.2,
            max_tokens=400
        ))
        print("programmer: done")


        extra_tests_raw = safe_api_call(
            messages=[{
                "role": "user",
                "content": f"""
You are a test designer agent.
Generate 2 assert statements.

Problem:
{problem}

Function name: {func_name}

Return ONLY assert statements, one per line.
"""
            }],
            temperature=0.3,
            max_tokens=150
        )
        print("test designer: done")

        # بنشغل على الـ original tests بس
        for attempt in range(retries):
            try:
                run_tests(code, test_list)
                print(f"PASSED (attempt {attempt + 1})")
                return True
            except Exception as e:
                print(f"FAIL (attempt {attempt + 1}): {str(e)[:60]}")
                if attempt < retries - 1:
                    code = extract_code(safe_api_call(
                        messages=[{
                            "role": "user",
                            "content": f"""
You are a programmer agent.
Fix the following code.

Problem:
{problem}

IMPORTANT: The function MUST be named exactly: {func_name}

Code:
{code}

Error:
{str(e)}

Return ONLY the corrected Python function.
"""
                        }],
                        temperature=0.2,
                        max_tokens=400
                    ))
                    print("programmer fix: done")

        return False

    except Exception as e:
        print(f"CRITICAL ERROR: {e}")
        return False


acc_ref_agentcoder = evaluate_method(ref_agentcoder_solve, "AgentCoder [19]", delay=30)
save_result("ref_agentcoder", acc_ref_agentcoder)


EVALUATING: AgentCoder [19]

[1/30] remove_Occ
programmer: done
test designer: done
PASSED (attempt 1)
PASS (1/1)

[2/30] sort_matrix
programmer: done
test designer: done
PASSED (attempt 1)
PASS (2/2)

[3/30] count_common
programmer: done
test designer: done
FAIL (attempt 1): 'list' object has no attribute 'keys'
programmer fix: done
FAIL (attempt 2): 
FAIL (2/3)

[4/30] find_Volume
programmer: done
test designer: done
PASSED (attempt 1)
PASS (3/4)

[5/30] split_lowerstring
programmer: done
test designer: done
FAIL (attempt 1): 
programmer fix: done
FAIL (attempt 2): 
FAIL (3/5)

[6/30] text_lowercase_underscore
programmer: done
test designer: done
FAIL (attempt 1): 
programmer fix: done
FAIL (attempt 2): 
FAIL (3/6)

[7/30] square_perimeter
programmer: done
test designer: done
PASSED (attempt 1)
PASS (4/7)

[8/30] remove_dirty_chars
programmer: done
test designer: done
PASSED (attempt 1)
PASS (5/8)

[9/30] test_duplicate
programmer: done
test designer: done
PASSED (attempt 1)
PASS (6

In [ ]:
# Codex Baseline Reference [9]
def ref_codex_solve(problem, test_list, func_name):
    try:
        code = extract_code(safe_api_call(
            messages=[{
                "role": "user",
                "content": f"{problem}\n# Function name: {func_name}"
            }],
            temperature=0.0,
            max_tokens=400
        ))
        run_tests(code, test_list)
        print("PASSED")
        return True
    except Exception as e:
        print(f"FAIL: {str(e)[:60]}")
        return False


acc_ref_codex = evaluate_method(ref_codex_solve, "Codex Baseline [9]", delay=10)
save_result("ref_codex", acc_ref_codex)


EVALUATING: Codex Baseline [9]

[1/30] remove_Occ
Helo Word
Pytho is fu
Test string
PASSED
PASS (1/1)

[2/30] sort_matrix
[1, 2, 3]
[4, 5, 6]
[3, 6, 9]
PASSED
PASS (2/2)

[3/30] count_common
{'apple': 5, 'orange': 5}
FAIL: 
FAIL (2/3)

[4/30] find_Volume
FAIL: find_Volume() takes 2 positional arguments but 3 were given
FAIL (2/4)

[5/30] split_lowerstring
['H', 'e', 'l', 'l', 'oW', 'o', 'r', 'l', 'd']
FAIL: 
FAIL (2/5)

[6/30] text_lowercase_underscore
['this_is', 'a_test', 'another_test']
FAIL: 
FAIL (2/6)

[7/30] square_perimeter
PASSED
PASS (3/7)

[8/30] remove_dirty_chars
Heo, ord!
PASSED
PASS (4/8)

[9/30] test_duplicate
PASSED
PASS (5/9)

[10/30] is_woodall
True
False
PASSED
PASS (6/10)

[11/30] multiples_of_num
FAIL: 
FAIL (6/11)

[12/30] find_first_duplicate
FAIL: 
FAIL (6/12)

[13/30] maximum_Sum
Maximum sum: 24
PASSED
PASS (7/13)

[14/30] binary_to_decimal
FAIL: 'int' object is not iterable
FAIL (7/14)

[15/30] find_Product
FAIL: find_Product() takes 1 positional argument bu

In [ ]:
#  MGDebugger Reference [6]
def ref_mgdebugger_solve(problem, test_list, func_name, retries=3):
    try:
        code = extract_code(safe_api_call(
            messages=[{
                "role": "user",
                "content": f"""
Write a correct Python function for the following problem.

Problem:
{problem}

IMPORTANT: The function MUST be named exactly: {func_name}

Return ONLY the Python function.
"""
            }],
            temperature=0.2,
            max_tokens=400
        ))
        decomp = safe_api_call(
            messages=[{
                "role": "user",
                "content": f"""
Decompose the following problem into small sub-functions.
For each sub-function, describe what it does.

Problem:
{problem}

Return ONLY the decomposition. No code.
"""
            }],
            temperature=0.3,
            max_tokens=300
        )
        for attempt in range(retries):
            try:
                run_tests(code, test_list)
                print(f"PASSED (attempt {attempt + 1})")
                return True
            except Exception as e:
                print(f"FAIL (attempt {attempt + 1}): {str(e)[:60]}")
                code = extract_code(safe_api_call(
                    messages=[{
                        "role": "user",
                        "content": f"""
You are a hierarchical debugger.

Decomposition:
{decomp}

IMPORTANT: The function MUST be named exactly: {func_name}

Code:
{code}

Error:
{str(e)}

Return ONLY the corrected Python function.
"""
                    }],
                    temperature=0.2,
                    max_tokens=500
                ))
                decomp = safe_api_call(
                    messages=[{
                        "role": "user",
                        "content": f"""
Update the decomposition based on the failure.

Problem:
{problem}

Error:
{str(e)}

Return updated decomposition steps only.
"""
                    }],
                    temperature=0.2,
                    max_tokens=200
                )
        return False
    except Exception as e:
        print(f"CRITICAL ERROR: {e}")
        return False


acc_ref_mgdebugger = evaluate_method(ref_mgdebugger_solve, "MGDebugger [6]", delay=25)
save_result("ref_mgdebugger", acc_ref_mgdebugger)


EVALUATING: MGDebugger [6]

[1/30] remove_Occ
PASSED (attempt 1)
PASS (1/1)

[2/30] sort_matrix
PASSED (attempt 1)
PASS (2/2)

[3/30] count_common
FAIL (attempt 1): 'list' object has no attribute 'lower'
FAIL (attempt 2): 
FAIL (attempt 3): 'list' object has no attribute 'items'
FAIL (2/3)

[4/30] find_Volume
PASSED (attempt 1)
PASS (3/4)

[5/30] split_lowerstring
FAIL (attempt 1): 
FAIL (attempt 2): 
FAIL (attempt 3): 
FAIL (3/5)

[6/30] text_lowercase_underscore
FAIL (attempt 1): 
FAIL (attempt 2): 
FAIL (attempt 3): 
FAIL (3/6)

[7/30] square_perimeter
PASSED (attempt 1)
PASS (4/7)

[8/30] remove_dirty_chars
PASSED (attempt 1)
PASS (5/8)

[9/30] test_duplicate
PASSED (attempt 1)
PASS (6/9)

[10/30] is_woodall
PASSED (attempt 1)
PASS (7/10)

[11/30] multiples_of_num
PASSED (attempt 1)
PASS (8/11)

[12/30] find_first_duplicate
FAIL (attempt 1): 
FAIL (attempt 2): 
FAIL (attempt 3): 
FAIL (8/12)

[13/30] maximum_Sum
PASSED (attempt 1)
PASS (9/13)

[14/30] binary_to_decimal
FAIL (attem

In [ ]:
with open(RESULTS_PATH, "r") as f:
    final_results = json.load(f)

reference_scores = {
    "standard_ma":       "-",
    "standard_sd":       "-",
    "standard_hybrid":   "-",
    "codesim_ma":        "90.7% [7]",
    "codesim_sd":        "90.7% [7]",
    "codesim_hybrid":    "-",
    "normal_agentcoder": "-",
    "normal_codex":      "-",
    "normal_mgdebugger": "-",
    "ref_agentcoder":    "~75% [19]",
    "ref_codex":         "~55% [9]",
    "ref_mgdebugger":    "~85% [6]",
}

labels = {
    "standard_ma":       "Standard Multi-Agent",
    "standard_sd":       "Standard Self-Debug",
    "standard_hybrid":   "Standard Hybrid",
    "codesim_ma":        "CODESIM Multi-Agent",
    "codesim_sd":        "CODESIM Self-Debug",
    "codesim_hybrid":    "CODESIM Hybrid",
    "normal_agentcoder": "Normal AgentCoder",
    "normal_codex":      "Normal Codex Baseline",
    "normal_mgdebugger": "Normal MGDebugger",
    "ref_agentcoder":    "AgentCoder [19]",
    "ref_codex":         "Codex Baseline [9]",
    "ref_mgdebugger":    "MGDebugger [6]",
}

print("\n" + "=" * 70)
print("FINAL RESULTS - MBPP (n=30)")
print("=" * 70)
print(f"{'Method':<30} {'Our Result':>12} {'Reference':>15}")
print("-" * 60)
for key, label in labels.items():
    our = final_results.get(key, "N/A")
    ref = reference_scores.get(key, "-")
    if isinstance(our, float):
        print(f"{label:<30} {our:>10.1f}% {ref:>15}")
    else:
        print(f"{label:<30} {'N/A':>11} {ref:>15}")
print("=" * 70)


FINAL RESULTS - MBPP (n=30)
Method                           Our Result       Reference
------------------------------------------------------------
Standard Multi-Agent                 63.3%               -
Standard Self-Debug                  60.0%               -
Standard Hybrid                      60.0%               -
CODESIM Multi-Agent                  83.3%       90.7% [7]
CODESIM Self-Debug                   90.0%       90.7% [7]
CODESIM Hybrid                       90.0%               -
Normal AgentCoder                    56.7%               -
Normal Codex Baseline                56.7%               -
Normal MGDebugger                    56.7%               -
AgentCoder [19]                      60.0%       ~75% [19]
Codex Baseline [9]                   50.0%        ~55% [9]
MGDebugger [6]                       63.3%        ~85% [6]
